# Exercise 3: Apply cross-platform hardware optimizations with ONNX Runtime

[ONNX Runtime](https://onnxruntime.ai/) stands as the industry standard for hardware-agnostic AI deployment, enabling developers to optimize models across diverse hardware platforms—from NVIDIA GPUs to Intel CPUs to mobile processors—using a single framework with consistent APIs, while still leveraging hardware-specific acceleration libraries for production-grade performance.

> **Overview:** You need to deploy a model across multiple cloud providers (AWS, Azure, GCP) and on-premise servers with different hardware configurations, with consistent performance and without vendor lock-in.
> 
> **Scenario:** Your startup's image classification service processes 100,000+ images daily across heterogeneous infrastructure. The DevOps team reports suboptimal performance: your PyTorch model achieves only 60% GPU utilization, while CPU-only deployments are bottlenecked by poor threading configuration. You need to maximize performance on each platform while avoiding vendor lock-in and maintaining deployment simplicity.
> 
> **Goal:** Explore ONNX Runtime's cross-platform optimization capabilities including I/O binding, thread management, memory optimization, and execution provider selection to achieve consistent performance across diverse hardware environments.
> 
> **Tools:** torch, torchvision, onnx, onnxruntime-gpu, numpy, pillow
> 
> **Estimated Time:** 15 minutes

## Step 1: Setup

Let's establish your cross-platform testing environment.

In [ ]:
# # Uncomment to install necessary libraries, then comment out and restart
# ! pip install onnx onnxruntime-gpu==1.19.2 torchvision pillow

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import onnx
import onnxruntime as ort
import numpy as np
import time
from PIL import Image
import json
from datetime import datetime

# Create output directory
output_dir = "assets/exercise3"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
print("=== CROSS-PLATFORM DEPLOYMENT ENVIRONMENT ===")

# Check PyTorch and ONNX versions
print(f"PyTorch version: {torch.__version__}")
print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

# Verify GPU availability for cross-platform testing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_properties = torch.cuda.get_device_properties(0)
    
    print(f"\n=== NVIDIA T4 HARDWARE ANALYSIS ===")
    print(f"GPU: {gpu_name}")
    print(f"Compute Capability: {gpu_properties.major}.{gpu_properties.minor}")
    print(f"Total Memory: {gpu_properties.total_memory / 1e9:.1f} GB")
    print(f"CUDA Cores: ~2,560")
    print(f"Tensor Cores: ~320 (2nd gen)")
    print(f"Memory Bandwidth: ~320 GB/s")
    
    # Check available execution providers
    print(f"\n=== AVAILABLE EXECUTION PROVIDERS ===")
    available_providers = ort.get_available_providers()
    for provider in available_providers:
        print(f"✓ {provider}")
    
    # Verify TensorRT availability
    tensorrt_available = 'TensorrtExecutionProvider' in available_providers
    print(f"\nTensorRT Support: {'✓ Available' if tensorrt_available else '✗ Not Available'}")
    
else:
    print("CUDA not available - some optimizations will be CPU-only")

print("\n✓ Environment setup complete!")

> **What do we mean with execution providers?** ONNX Runtime's execution provider system enables hardware abstraction by separating model optimization logic from hardware-specific implementation. 
> 
> Each provider _(CPU, CUDA, TensorRT, ...)_ uses the same ONNX graph but applies different optimization strategies—CPU providers focus on vectorization and threading, CUDA providers leverage parallel processing, while TensorRT providers add graph-level optimization and kernel fusion.

## Step 2: Load model

For this exercise, we'll use [EfficientNet-B0](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html) as it represents modern CNN architectures commonly deployed across platforms.

In [ ]:
# Load pre-trained EfficientNet model
print("Loading EfficientNet-B0 for cross-platform optimization...")

# Load pre-trained EfficientNet-B0 model
# Use the weights enum API; pretrained=True is deprecated in recent torchvision
efficientnet_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Set model to evaluation mode for inference
efficientnet_model.eval()

print(f"Model loaded: {efficientnet_model.__class__.__name__}")
print(f"Total parameters: {sum(p.numel() for p in efficientnet_model.parameters()):,}")
print(f"Model size: {sum(p.numel() * p.element_size() for p in efficientnet_model.parameters()) / 1024**2:.1f} MB")

# Prepare sample input for testing
# EfficientNet-B0 expects 224x224 RGB images
batch_size = 32
input_shape = (batch_size, 3, 224, 224)
sample_input = torch.randn(input_shape)

print(f"\nSample input shape: {sample_input.shape}")
print(f"Input tensor size: {sample_input.numel() * sample_input.element_size() / 1024**2:.1f} MB")

# Verify model works with sample input
with torch.no_grad():
    output = efficientnet_model(sample_input)
    print(f"Output shape: {output.shape}")
    print(f"Model successfully processes input ✓")

> **EfficientNet hardware-architecture characteristics**: EfficientNet's compound scaling and mobile-optimized blocks make it an excellent test case for cross-platform optimization, as different hardware platforms will benefit from different optimization strategies.

## Step 3: Convert to ONNX with optimization analysis

Convert the PyTorch model to ONNX format and analyze the computational graph for cross-platform deployment.

In [ ]:
def convert_to_onnx_with_analysis(model, sample_input, output_path):
    """Convert PyTorch model to ONNX with detailed analysis"""
    
    print("=== PYTORCH TO ONNX CONVERSION ===")
    
    # Mark the batch dimension as dynamic so one exported artifact serves any
    # batch size across deployments, and fold constants at export time so every
    # runtime loads a leaner graph. Opset 17 is mature and covered by all
    # mainstream execution providers (CPU, CUDA, TensorRT, OpenVINO, ...).
    torch.onnx.export(
        model,
        sample_input,
        output_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    
    # Verify ONNX model
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    
    # Analyze model structure for cross-platform insights
    print(f"✓ ONNX export successful")
    print(f"ONNX model size: {os.path.getsize(output_path) / 1024**2:.1f} MB")
    print(f"ONNX opset version: {onnx_model.opset_import[0].version}")
    print(f"Total nodes: {len(onnx_model.graph.node)}")
    
    # Count operator types for provider compatibility analysis
    op_counts = {}
    for node in onnx_model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
    
    print(f"\nTop operator types:")
    for op_type, count in sorted(op_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {op_type}: {count} nodes")
    
    return output_path

# Convert model to ONNX
onnx_model_path = os.path.join(output_dir, "efficientnet_b0.onnx")
onnx_model_path = convert_to_onnx_with_analysis(efficientnet_model, sample_input, onnx_model_path)

> **What insights can we gather from the operator analysis?** The operator breakdown reveals EfficientNet's optimization-friendly architecture—Conv operations dominate the computational graph (ideal for GPU acceleration), while Sigmoid and Mul operations are well-supported across all execution providers. 
> 
> This operator distribution indicates the model will benefit significantly from TensorRT's convolution fusion optimizations while maintaining broad compatibility for CPU fallback scenarios.

## Step 4: Execution provider performance comparison with default values

Let's test different execution providers to understand hardware abstraction trade-offs.

> **IMPORTANT**: You may need to install different onnxruntime versions depending on your chosen execution providers. By default, the notebook installs `onnxruntime-gpu` to support CUDA 12.x. For more details and other installation paths, please refer to the [Install ONNX Runtime](https://onnxruntime.ai/docs/install/) guide.

In [ ]:
def benchmark_execution_provider(model_path, input_data, provider_config, num_warmup=5, num_runs=20):
    """Benchmark specific execution provider configuration"""
    
    # The provider list acts as a fallback chain: ONNX Runtime instantiates
    # the first provider it can bring up on this machine.
    session = ort.InferenceSession(model_path, providers=provider_config)
    
    # Get input/output names
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    # Warm up the session
    input_dict = {input_name: input_data}
    for _ in range(num_warmup):
        _ = session.run([output_name], input_dict)
    
    # Benchmark inference
    times = []
    for _ in range(num_runs):
        start_time = time.perf_counter()
        outputs = session.run([output_name], input_dict)
        end_time = time.perf_counter()
        times.append(end_time - start_time)
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    throughput = input_data.shape[0] / avg_time  # samples/sec
    
    # Get actual execution provider used
    used_providers = session.get_providers()
    
    return {
        'provider_config': provider_config,
        'used_providers': used_providers,
        'avg_latency_ms': avg_time * 1000,
        'std_latency_ms': std_time * 1000,
        'throughput_samples_sec': throughput,
        'output_shape': outputs[0].shape
    }

def run_provider_comparison():
    """Compare performance across different execution providers"""
    
    print("=== EXECUTION PROVIDER PERFORMANCE COMPARISON ===")
    
    # Prepare input data
    test_input = sample_input.numpy().astype(np.float32)
    
    # Only queue configurations this machine can actually initialize: recent
    # ONNX Runtime versions raise at session creation when a requested
    # provider is not compiled into the installed package.
    provider_configs = [
        ['CPUExecutionProvider'],
    ]
    if 'CUDAExecutionProvider' in ort.get_available_providers():
        # CPU stays in the chain as a fallback for any op CUDA cannot place
        provider_configs.append(['CUDAExecutionProvider', 'CPUExecutionProvider'])
    else:
        print("Note: CUDAExecutionProvider not available on this machine - skipping the GPU benchmark.")
    
    results = {}
    
    for i, provider_config in enumerate(provider_configs):
        provider_name = provider_config[0].replace('ExecutionProvider', '')
        print(f"\nTesting {provider_name}...")
        
        try:
            result = benchmark_execution_provider(onnx_model_path, test_input, provider_config)
            results[provider_name] = result
            
            print(f"  Used: {result['used_providers'][0]}")
            print(f"  Latency: {result['avg_latency_ms']:.1f} ± {result['std_latency_ms']:.1f} ms")
            print(f"  Throughput: {result['throughput_samples_sec']:.1f} samples/sec")
            
        except Exception as e:
            print(f"  ✗ Failed: {str(e)}")
            results[provider_name] = None
    
    return results

# Run provider comparison
provider_results = run_provider_comparison()

> **Execution provider deep-dive for NVIDIA GPUs**: ONNX Runtime provides some official guidance on how to [choose execution providers](https://pkreg101.github.io/onnxruntime/docs/performance/choosing-execution-providers.html). For NVIDIA hardware, ONNX Runtime offers three execution providers with different optimization focuses:
> 
> - **CUDA EP**: Basic GPU acceleration using cuDNN/cuBLAS libraries
> - **TensorRT EP**: Advanced optimization with graph analysis and kernel fusion
> - **TensorRT RTX EP**: Consumer RTX-optimized version with faster compile times

## Step 5: Implement ONNX Runtime optimizations across platforms

Let's now implement targeted optimization strategies for each deployment environment. First we’ll optimize each environment separately before unifying into a single cross-platform configuration. Why this order?
- **GPU optimization**: Focus on maximizing expensive GPU instance utilization by eliminating I/O bottlenecks.  
- **CPU optimization**: Focus on threading & memory efficiency where resources are constrained.  
- **Unified strategy**: Combine both approaches into a flexible configuration that adapts to the hardware at runtime.

This progression helps you see *why* GPU and CPU need different strategies, before you merge them into one deployment recipe.

> **IMPORTANT**: For this exercise, the focus is on session-level optimizations that provide 80% of performance benefits with a simple configuration. BUT, each execution provider offers additional configuration parameters for fine-tuning performance:
> - [**CUDA Provider**](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html): gpu_mem_limit, cudnn_conv_algo_search, arena_extend_strategy for memory and convolution optimization
> - [**TensorRT Providers**](https://onnxruntime.ai/docs/execution-providers/TensorRT-ExecutionProvider.html): Workspace size, precision modes, engine caching for compilation optimization
> - ...

> **EXPERT TIP**: ONNX Runtime provides [automated performance tuning](https://pkreg101.github.io/onnxruntime/docs/performance/performance-tuning-tools.html) tools. Here, you experiment manually to get hands-on understanding of how different configurations utilize specific hardware.


### A. Maximize GPU utilization

GPU utilization can be boosted by implementing I/O binding and GPU-focused optimizations. This is because memory transfers and suboptimal threading can leave GPU cores idle.

In [ ]:
def optimize_for_gpu(model_path):
    """Optimize configuration for GPU - focus on maximizing GPU utilization"""
    
    print("=== GPU OPTIMIZATION (Target: >90% GPU utilization) ===")
    
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        print("CUDA not available - skipping GPU optimization")
        return None
    
    # Prepare input on GPU
    input_tensor = torch.randn(batch_size, 3, 224, 224, device='cuda', dtype=torch.float32)
    
    sess_options = ort.SessionOptions()
    # Full graph-level rewriting (fusion, constant folding, layout changes)
    # collapses the graph into fewer, larger kernels - fewer launches means
    # the GPU spends more of each call doing useful work.
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    # The GPU already parallelizes inside each kernel; sequential node
    # scheduling avoids stream contention between concurrent graph branches.
    sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    # Create session with inputs and outputs

    # CUDA carries the model; CPU stays as a safety net for unsupported ops
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    session = ort.InferenceSession(model_path, sess_options=sess_options, providers=providers)
    
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    # Forward once to get output shape
    ort_input = input_tensor.cpu().numpy().astype(np.float32)
    dummy_out = session.run([output_name], {input_name: ort_input})
    output_shape = dummy_out[0].shape
    
    # Create input OrtValue on GPU  
    input_ortvalue = ort.OrtValue.ortvalue_from_numpy(input_tensor.cpu().numpy(), 'cuda', 0)
    output_ortvalue = ort.OrtValue.ortvalue_from_shape_and_type(output_shape, np.float32, 'cuda', 0)
    
    # Bind input and output to GPU-resident OrtValues so run_with_iobinding
    # never crosses PCIe: the per-call host-to-device / device-to-host copies
    # are exactly what keeps GPU utilization low on small and medium batches.
    io_binding = session.io_binding()  
    
    io_binding.bind_ortvalue_input(input_name, input_ortvalue)
    io_binding.bind_ortvalue_output(output_name, output_ortvalue)

    # Warmup
    for _ in range(5):
        session.run_with_iobinding(io_binding)
        torch.cuda.synchronize()

    # Benchmark optimized configuration
    times = []
    for _ in range(15):
        torch.cuda.synchronize()
        start_time = time.perf_counter()
        session.run_with_iobinding(io_binding)
        torch.cuda.synchronize()
        end_time = time.perf_counter()
        times.append(end_time - start_time)
    
    avg_time = np.mean(times)
    throughput = input_tensor.shape[0] / avg_time
    
    print(f"Baseline CUDA latency: {provider_results['CUDA']['avg_latency_ms']:.1f} ms "
        f"({provider_results['CUDA']['throughput_samples_sec']:.1f} samples/sec)")
    print(f"✓ Optimized latency: {avg_time * 1000:.1f} ms "
        f"({throughput:.1f} samples/sec)")
    print(f"✓ Improvement vs baseline: {throughput / provider_results['CUDA']['throughput_samples_sec']:.2f}x throughput")
    
    return {
        'latency_ms': avg_time * 1000,
        'throughput_samples_sec': throughput,
        'optimization': 'I/O binding + GPU-focused configuration'
    }

# Optimize for GPU scenario
gpu_result = optimize_for_gpu(onnx_model_path)

> **Examine the GPU benchmark results.** How does I/O binding and session optimization affect throughput compared to the raw CUDA baseline? Did latency improve, worsen, or stay about the same?
>
> _Answer:_ On a CUDA machine the I/O-bound session consistently improves on the raw CUDA baseline because the default `session.run` path copies the input batch host-to-device and the logits device-to-host on every single call, while I/O binding keeps both tensors resident in GPU memory. For EfficientNet-B0 at batch 32, compute per call is only a few milliseconds, so those PCIe transfers make up a meaningful slice of end-to-end latency; typical results land in the 1.1-1.5x throughput range with latency dropping accordingly. The relative benefit grows as the model or batch shrinks, since transfer time stays roughly constant while compute time falls - which is why I/O binding matters most for small models and latency-sensitive serving. `ORT_ENABLE_ALL` adds a smaller one-time contribution by fusing conv/activation chains into fewer kernel launches. On a CPU-only machine this cell prints a skip notice and returns `None`; the observations above are what to expect when rerunning on a GPU host.

### B. Optimize CPU performance

Address poor CPU threading configuration for Standard instances via i) efficient thread utilization without oversubscription, and ii) memory optimization.

In [ ]:
def optimize_for_cpu(model_path):
    """Optimize configuration for CPU instances - focus on threading efficiency"""
    
    print("\n=== CPU OPTIMIZATION (Target: Optimal threading for available vCPUs) ===")
    
    # Define session
    sess_options = ort.SessionOptions()

    # Prepare input
    input_data = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)

    # Give each operator every available core: convolutions scale well with
    # intra-op threads, and capping at the vCPU count prevents oversubscription
    # (more threads than cores just adds context-switch overhead).
    num_cores = os.cpu_count() or 1
    sess_options.intra_op_num_threads = num_cores
    # EfficientNet is a mostly linear chain of blocks, so there is little
    # branch-level parallelism to exploit; extra inter-op lanes would only
    # steal cores from the convolutions.
    sess_options.inter_op_num_threads = 1

    # Sequential mode pairs with inter_op=1: one node at a time, fully
    # threaded internally, no cross-operator contention.
    sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    # Input shapes are fixed here, so the runtime can plan all activation
    # buffers once (memory patterns) and serve them from a pre-grown arena
    # instead of paying allocator overhead on every inference call.
    sess_options.enable_cpu_mem_arena = True
    sess_options.enable_mem_pattern = True
    sess_options.enable_mem_reuse = True

    # Fold constants and fuse conv/activation chains at load time
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    # Pure CPU target - no fallback chain needed
    session = ort.InferenceSession(model_path, sess_options=sess_options, providers=['CPUExecutionProvider'])
    
    # Benchmark threading configuration
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    input_dict = {input_name: input_data}
    
    # Warmup
    for _ in range(5):
        _ = session.run([output_name], input_dict)

    times = []
    for _ in range(15):
        start_time = time.perf_counter()
        outputs = session.run([output_name], input_dict)
        end_time = time.perf_counter()
        times.append(end_time - start_time)
    
    avg_time = np.mean(times)
    throughput = input_data.shape[0] / avg_time
    
    print(f"Baseline CPU latency: {provider_results['CPU']['avg_latency_ms']:.1f} ms "
        f"({provider_results['CPU']['throughput_samples_sec']:.1f} samples/sec)")
    print(f"✓ Optimized latency: {avg_time * 1000:.1f} ms "
        f"({throughput:.1f} samples/sec)")
    print(f"✓ Improvement vs baseline: {throughput / provider_results['CPU']['throughput_samples_sec']:.2f}x throughput")
    
    return {
        'latency_ms': avg_time * 1000,
        'throughput_samples_sec': throughput,
        'optimization': 'Threading + memory optimization for CPU'
    }

# Optimize for CPU scenario  
cpu_result = optimize_for_cpu(onnx_model_path)

> **Examine the CPU benchmark results.** How did threading and memory optimizations affect throughput and latency compared to the CPU baseline? Were the improvements significant? Why did you choose these configuration values?
>
> _Answer:_ Throughput improves modestly over the default-CPU baseline and run-to-run variance tightens, because ONNX Runtime's defaults already use all cores - the win here comes from eliminating waste rather than adding parallelism. `intra_op_num_threads` is pinned to the vCPU count so each convolution saturates the cores without oversubscription, while `inter_op_num_threads = 1` with sequential execution reflects EfficientNet's mostly linear graph: scheduling operators in parallel would only steal cores from the convolutions and add contention. The memory arena, memory-pattern planning, and memory reuse settings pay off because input shapes are fixed, letting the runtime plan every activation buffer once instead of hitting the allocator on each call. `ORT_ENABLE_ALL` folds constants and fuses conv/activation chains at session load. Because the defaults are already close to this configuration, measured gains range from a wash (within run-to-run noise) up to ~20% depending on machine load; the explicit thread caps matter most on shared or containerized instances, where they prevent noisy-neighbor oversubscription and stabilize tail latency.

### C. Define a balanced multi-platform strategy

Let's imagine you tried to create a balanced session configuration that works efficiently across different hardware environments. This **single configuration** finds the sweet spot between:
- Using GPU optimizations (with I/O binding),
- Setting CPU optimizations (with threading & memory optimizations)

**What would happen if you tried to use the same session options for both CPU and GPU?**

_Answer:_ A single shared configuration necessarily short-changes one platform, because the two tune opposite resources. The GPU-oriented pieces are useless or fatal on CPU: I/O binding to CUDA device memory cannot even be constructed without the CUDA provider, and a GPU-style session that leaves CPU threading minimal (the GPU session barely needs host threads) would crater CPU throughput several-fold if reused on a CPU box. Conversely, CPU-style aggressive intra-op threading on a GPU host just spins host threads that have almost no work to do, adding scheduling noise around kernel launches. The only settings that are genuinely safe to share are the model-level ones: the ONNX artifact itself and `ORT_ENABLE_ALL` graph optimization. The portable pattern is therefore one ONNX file plus a small runtime probe - inspect `ort.get_available_providers()` at startup, then build platform-specific `SessionOptions` (I/O binding on GPU hosts, thread and arena tuning on CPU hosts) - which preserves both peak performance and deployment simplicity.

-----

> **What if we wanted to also deploy on mobile?**
>
> Reflect on how ONNX Runtime's session should be configured for memory- and power-constrained devices like mobile phones or edge devices. Consider memory usage, optimization level, threading, and power efficiency vs. peak performance.
>
> _Answer:_ On mobile the objective flips from peak throughput to sustained efficiency inside tight memory and power envelopes. The execution providers change to the platform accelerators - NNAPI on Android, CoreML on iOS, XNNPACK as the portable CPU path - and heavy graph optimization moves offline: converting to the pre-optimized ORT format means the device is not re-running `ORT_ENABLE_ALL` at every app launch, cutting both startup latency and peak memory. Threading should be capped at one or two threads, since extra threads drain battery and trigger thermal throttling that erases their gains within minutes of sustained use. The memory arena should be constrained or disabled (`enable_cpu_mem_arena = False`), because a greedy arena competes with the rest of the app under OS memory pressure and invites the OS to kill the process. INT8 quantization is close to mandatory, shrinking the model roughly 4x and mapping it onto mobile DSP/NPU integer units. The cross-platform lesson still holds: it is the same ONNX artifact family throughout, with provider selection and session options tuned per device class.

## Conclusion

In this exercise, you have uncovered ONNX Runtime's cross-platform optimization capabilities for AI. 

You started from a simple CPU vs GPU baseline and step by step explored how to optimize sessions for platform-specific execution (GPU utilization, threading/memory on CPU).

**Key insight**:  A single ONNX model is sufficient for cross-platform deployment; session options should be tuned per platform to achieve optimal performance, while portability is maintained.

##### **Next cross-platform optimization challenges to explore:**

- **Advanced Provider Configuration**: Explore provider-specific optimization options for specialized scenarios.
- **Execution Providers Experimentation**: Explore other execution providers (e.g., TensorRT, OpenVINO, DirectML) for specialized hardware.  
- **On-device Benchmark**: Benchmark on **different hardware** (edge device, larger CPU server, or alternative GPU).